# Grok-gpu-t4x2-smoke

> **Domain:** gpu · **Task:** t4x2-smoke
>
> Confirm Kaggle **T4×2** is visible to PyTorch, run dual-GPU GEMM + DataParallel mini-train,
> and write a machine-readable result under `/kaggle/working`.

Naming: `Grok-{领域}-{任务}` → `Grok-gpu-t4x2-smoke`.

In [ ]:
# -*- env: make both T4s visible -*-
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("CUDA_VISIBLE_DEVICES unset → use all visible GPUs")

In [ ]:
# -*- setup + device probe -*-
import json, time, platform, random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.parallel import DataParallel

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available — attach GPU accelerator"

n = torch.cuda.device_count()
devices = []
for i in range(n):
    name = torch.cuda.get_device_name(i)
    props = torch.cuda.get_device_properties(i)
    info = {
        "index": i,
        "name": name,
        "mem_gb": round(props.total_memory / 1e9, 2),
        "capability": list(torch.cuda.get_device_capability(i)),
    }
    devices.append(info)
    print(f"  [{i}] {name}  {info['mem_gb']}GB  cap={info['capability']}")

assert n >= 1, "Need at least 1 GPU"
is_t4 = all("T4" in d["name"] for d in devices)
print(f"device_count={n} is_t4={is_t4}")
if n < 2:
    print("WARN: expected T4×2 (2 devices); continuing with available GPUs")

device = torch.device("cuda:0")

In [ ]:
# -*- dual-GPU FP16 GEMM micro-benchmark -*-
N = 4096
iters = 10
gemm_results = []

for i in range(n):
    dev = torch.device(f"cuda:{i}")
    a = torch.randn(N, N, device=dev, dtype=torch.float16)
    b = torch.randn(N, N, device=dev, dtype=torch.float16)
    for _ in range(3):
        c = a @ b
    torch.cuda.synchronize(dev)
    t0 = time.perf_counter()
    for _ in range(iters):
        c = a @ b
    torch.cuda.synchronize(dev)
    elapsed = time.perf_counter() - t0
    flops = 2 * (N ** 3) * iters
    tflops = flops / elapsed / 1e12
    row = {"gpu": i, "n": N, "iters": iters, "seconds": round(elapsed, 4), "tflops_fp16": round(tflops, 3)}
    gemm_results.append(row)
    print(f"GPU{i} GEMM {N}x{N} x{iters}: {elapsed:.3f}s ~{tflops:.2f} TFLOPS (FP16)")

del a, b, c
torch.cuda.empty_cache()

In [ ]:
# -*- tiny CNN + DataParallel on all visible GPUs -*-
class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)


n_samples = 2048
x = torch.randn(n_samples, 3, 32, 32)
y = torch.randint(0, 10, (n_samples,))
loader = DataLoader(TensorDataset(x, y), batch_size=256 if n >= 2 else 128, shuffle=True, num_workers=0)

model = TinyCNN().to(device)
if n >= 2:
    model = DataParallel(model)
    print(f"DataParallel on {n} GPUs")
else:
    print("single GPU (no DataParallel)")

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
history = []
epochs = 5

t0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    avg = total_loss / total
    acc = correct / total
    history.append({"epoch": epoch, "loss": round(avg, 5), "acc": round(acc, 4)})
    print(f"epoch {epoch}/{epochs}  loss={avg:.4f}  acc={acc:.3f}")

train_seconds = round(time.perf_counter() - t0, 3)
print(f"train_seconds={train_seconds}")
assert history[-1]["loss"] < history[0]["loss"] or history[-1]["acc"] > history[0]["acc"], \
    "training did not improve — check GPU kernels"

params = sum(p.numel() for p in (model.module if isinstance(model, DataParallel) else model).parameters())
print("params", params)

In [ ]:
# -*- write results -*-
result = {
    "notebook": "Grok-gpu-t4x2-smoke",
    "domain": "gpu",
    "task": "t4x2-smoke",
    "status": "ok",
    "torch": torch.__version__,
    "python": platform.python_version(),
    "cuda_available": True,
    "device_count": n,
    "devices": devices,
    "is_t4": is_t4,
    "dual_gpu": n >= 2,
    "gemm": gemm_results,
    "train_seconds": train_seconds,
    "history": history,
    "params": params,
    "epochs": epochs,
}

path = OUT / "grok_gpu_t4x2_smoke_results.json"
path.write_text(json.dumps(result, indent=2))
print("wrote", path)
print(json.dumps(result, indent=2))

# Lightweight checkpoint for output browser
torch.save(
    {
        "state_dict": (model.module if isinstance(model, DataParallel) else model).state_dict(),
        "history": history,
    },
    OUT / "tiny_cnn_dp.pt",
)
print("done ✓")